# NIH ChestX-ray14 — Patient-grouped split & Dataset

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_ROOT  = Path(r"C:\Users\mamou\.cache\kagglehub\datasets\nih-chest-xrays\data\versions\3")
IMAGES_DIR = DATA_ROOT / "images"                                          # unified folder (merged in nb 01)
OUT_DIR    = Path(r"d:\My Projects\chest-xray-bench\data\chestx_ray14")    # where split CSVs go
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 5 target tasks, shared with CheXpert. NIH 'Effusion' == CheXpert 'Pleural Effusion'.
TARGET_TASKS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion"]

# --- Load + clean the master metadata (same steps as notebook 01) ---
df = pd.read_csv(DATA_ROOT / "Data_Entry_2017.csv").rename(columns={
    "OriginalImage[Width": "OriginalWidth", "Height]": "OriginalHeight",
    "OriginalImagePixelSpacing[x": "PixelSpacing_x", "y]": "PixelSpacing_y",
}).drop(columns=["Unnamed: 11"])

# Build binary 0/1 columns for the 5 target tasks from the pipe-separated Finding Labels
finding_sets = df["Finding Labels"].str.split("|")
for t in TARGET_TASKS:
    df[t] = finding_sets.apply(lambda s: int(t in s)).astype("int8")

# Tag the official split (patient-disjoint train_val / test)
train_val_set = set((DATA_ROOT / "train_val_list.txt").read_text().split())
test_set      = set((DATA_ROOT / "test_list.txt").read_text().split())
df["split"] = np.where(df["Image Index"].isin(train_val_set), "train_val", "test")

print("rows:", len(df), "| columns:", list(df.columns))
print(df["split"].value_counts().to_dict())
df.head()

rows: 112120 | columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalWidth', 'OriginalHeight', 'PixelSpacing_x', 'PixelSpacing_y', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'split']
{'train_val': 86524, 'test': 25596}


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalWidth,OriginalHeight,PixelSpacing_x,PixelSpacing_y,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,split
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,0,1,0,0,0,train_val
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,0,1,0,0,0,train_val
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,0,1,0,0,1,train_val
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,0,0,0,0,0,train_val
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,0,0,0,0,0,test


In [3]:
# Carve a patient-grouped val out of train_val (frozen, reproducible). test stays untouched.
from sklearn.model_selection import GroupShuffleSplit

VAL_FRAC, SEED = 0.05, 42   # 5% val: this is pretraining, keep as much train as possible

trainval = df[df["split"] == "train_val"].copy()

def prev(d):  # %positive per task
    return {t: round(100 * d[t].mean(), 2) for t in TARGET_TASKS}

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
tr_idx, va_idx = next(gss.split(trainval, groups=trainval["Patient ID"]))
tr = trainval.iloc[tr_idx].copy()
va = trainval.iloc[va_idx].copy()

p_tr, p_va = set(tr["Patient ID"]), set(va["Patient ID"])
print(f"VAL_FRAC={VAL_FRAC}  SEED={SEED}")
print(f"train : {len(tr):>6d} rows ({100*len(tr)/len(trainval):.1f}%)  patients {len(p_tr)}")
print(f"val   : {len(va):>6d} rows ({100*len(va)/len(trainval):.1f}%)  patients {len(p_va)}")
print(f"shared patients (must be 0): {len(p_tr & p_va)}")
print(f"\ntrain %pos: {prev(tr)}")
print(f"val   %pos: {prev(va)}")

VAL_FRAC=0.05  SEED=42
train :  82257 rows (95.1%)  patients 26607
val   :   4267 rows (4.9%)  patients 1401
shared patients (must be 0): 0

train %pos: {'Atelectasis': np.float64(9.53), 'Cardiomegaly': np.float64(1.97), 'Consolidation': np.float64(3.29), 'Edema': np.float64(1.58), 'Effusion': np.float64(10.06)}
val   %pos: {'Atelectasis': np.float64(10.43), 'Cardiomegaly': np.float64(2.04), 'Consolidation': np.float64(3.33), 'Edema': np.float64(1.85), 'Effusion': np.float64(9.0)}


In [4]:
# Write the frozen split CSVs to data/chestx_ray14/  (01_train.csv, 01_val.csv, 01_test.csv)
test = df[df["split"] == "test"].copy()

KEEP_COLS = ["path", "Image Index", "Patient ID", "Patient Age", "Patient Gender",
             "View Position"] + TARGET_TASKS + ["split"]

def finalize(d, split_name):
    d = d.copy()
    d["split"] = split_name
    d["path"] = "images/" + d["Image Index"]   # relative to DATA_ROOT
    return d[KEEP_COLS]

outputs = {"01_train.csv": finalize(tr, "train"),
           "01_val.csv":   finalize(va, "val"),
           "01_test.csv":  finalize(test, "test")}

for fname, d in outputs.items():
    path = OUT_DIR / fname
    d.to_csv(path, index=False)
    chk = pd.read_csv(path)
    print(f"{fname:<14} {len(chk):>6d} rows  cols={len(chk.columns)}  "
          f"patients={chk['Patient ID'].nunique()}  match={len(chk)==len(d)}")

print("\ncolumns:", KEEP_COLS)
print("total written:", sum(len(d) for d in outputs.values()), "(== 112120:",
      sum(len(d) for d in outputs.values()) == len(df), ")")

01_train.csv    82257 rows  cols=12  patients=26607  match=True
01_val.csv       4267 rows  cols=12  patients=1401  match=True
01_test.csv     25596 rows  cols=12  patients=2797  match=True

columns: ['path', 'Image Index', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'split']
total written: 112120 (== 112120: True )


In [6]:
# Preprocess: grayscale -> [CLAHE] -> aspect-preserving resize + center zero-pad -> 3ch -> ImageNet-norm -> tensor
# Matches the CheXpert pipeline exactly. CLAHE off by default.
import cv2, torch
from PIL import Image

IMG_W, IMG_H  = 384, 320
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def resize_pad(img, out_w=IMG_W, out_h=IMG_H):
    h0, w0 = img.shape
    scale = min(out_w / w0, out_h / h0)
    new_w, new_h = max(1, round(w0 * scale)), max(1, round(h0 * scale))
    img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((out_h, out_w), dtype=img.dtype)
    top, left = (out_h - new_h) // 2, (out_w - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = img
    return canvas

def preprocess(rel_path, use_clahe=False):
    img = np.asarray(Image.open(DATA_ROOT / rel_path).convert("L"))   # forces RGBA->L too
    if use_clahe:
        img = _clahe.apply(img)
    img = resize_pad(img)
    img = np.stack([img, img, img], axis=-1).astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(img.transpose(2, 0, 1))                   # (C,H,W) float32

# Verify on a random train image
rel = "images/" + tr.sample(1)["Image Index"].iloc[0]
t = preprocess(rel)
print(f"image: {rel}")
print(f"shape {tuple(t.shape)}  dtype {t.dtype}  min {t.min():.3f}  max {t.max():.3f}  "
      f"per-ch mean {[round(float(t[c].mean()),3) for c in range(3)]}")
print("expected shape (3, 320, 384)")

image: images/00004122_000.png
shape (3, 320, 384)  dtype torch.float32  min -2.118  max 2.187  per-ch mean [0.171, 0.304, 0.525]
expected shape (3, 320, 384)


In [7]:
# Dataset: preprocess() + the 5 binary label columns -> (image_tensor, label_tensor)
from torch.utils.data import Dataset

class ChestXray14Dataset(Dataset):
    def __init__(self, df, tasks=TARGET_TASKS, use_clahe=False):
        self.paths     = df["path"].tolist()
        self.labels    = torch.from_numpy(df[tasks].to_numpy(dtype="float32"))  # (N, 5)
        self.tasks     = tasks
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = preprocess(self.paths[i], use_clahe=self.use_clahe)
        return img, self.labels[i]

# Load the saved split CSVs (these carry the 'path' column) and verify on a random item
train_df = pd.read_csv(OUT_DIR / "01_train.csv")
val_df   = pd.read_csv(OUT_DIR / "01_val.csv")

ds = ChestXray14Dataset(train_df)
i = np.random.randint(len(ds))
img, label = ds[i]
print(f"len {len(ds)}  | index {i}  | path {ds.paths[i]}")
print(f"img   -> shape {tuple(img.shape)}  dtype {img.dtype}  min {img.min():.3f}  max {img.max():.3f}")
print(f"label -> shape {tuple(label.shape)}  dtype {label.dtype}  values {label.tolist()}")
print(f"tasks -> {ds.tasks}")

len 82257  | index 32173  | path images/00010542_000.png
img   -> shape (3, 320, 384)  dtype torch.float32  min -2.118  max 2.501
label -> shape (5,)  dtype torch.float32  values [1.0, 0.0, 0.0, 0.0, 0.0]
tasks -> ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion']


In [9]:
# DataLoader batch sanity check (num_workers=0 for Windows/Jupyter; raise on Modal/Linux)
import time
from torch.utils.data import DataLoader

BATCH_SIZE = 64
train_ds = ChestXray14Dataset(train_df)
val_ds   = ChestXray14Dataset(val_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"train: {len(train_ds)} imgs -> {len(train_loader)} batches")
print(f"val  : {len(val_ds)} imgs -> {len(val_loader)} batches")

imgs, labels = next(iter(train_loader))
print(f"\none batch: imgs {tuple(imgs.shape)} {imgs.dtype}  labels {tuple(labels.shape)} {labels.dtype}")
print(f"per-task positives in batch: {labels.sum(0).tolist()}")

N = 5
t0 = time.time()
for k, _ in zip(range(N), train_loader):
    pass
dt = time.time() - t0
print(f"\nthroughput: {dt/N:.3f}s/batch over {N} batches  (~{BATCH_SIZE*N/dt:.0f} imgs/s, single-process)")

train: 82257 imgs -> 1286 batches
val  : 4267 imgs -> 67 batches

one batch: imgs (64, 3, 320, 384) torch.float32  labels (64, 5) torch.float32
per-task positives in batch: [6.0, 1.0, 1.0, 3.0, 9.0]

throughput: 1.520s/batch over 5 batches  (~42 imgs/s, single-process)


## Summary — splits & Dataset

- **Frozen patient-grouped split** of `train_val` (GroupShuffleSplit, `VAL_FRAC=0.05`, `SEED=42`): **train 82,257** / **val 4,267** imgs, 0 shared patients. Official **test 25,596** untouched.
- **CSVs written** to `data/chestx_ray14/`: `01_train.csv`, `01_val.csv`, `01_test.csv` — columns `path, Image Index, Patient ID, Patient Age, Patient Gender, View Position, <5 tasks>, split`.
- **5 target tasks** (binary, no uncertainty): Atelectasis, Cardiomegaly, Consolidation, Edema, Effusion.
- **`preprocess()`** = grayscale (`L`, fixes RGBA) → optional CLAHE (off) → aspect resize + zero-pad to **384×320** → 3ch → ImageNet-norm → `(3,320,384)` tensor. Identical to CheXpert.
- **`ChestXray14Dataset`** → `(image, 5-label)` ; verified with DataLoader.

→ Ready to pre-train the best model on NIH, then fine-tune on CheXpert (replace the 5-class head as needed).